# AI와 함께 타이타닉 데이터 분석 한 사이클 완주하기

STEP 00–17 학생용 실행 Notebook입니다.

핵심 원칙은 **셀을 위에서 아래로 직접 실행하면서 데이터가 어떻게 변하는지 확인하는 것**입니다. 모델링 단계에서도 결측치 처리, 인코딩, 정규화/표준화, 모델 학습을 한꺼번에 묶지 않고 각각 따로 실행합니다.


## STEP 00. 전체 분석 지도

환경 → 데이터 → 품질 → Target → 결측/컬럼/인코딩 원리 → 시각화 → EDA/통계 → Feature → split → 결측치 처리 → 인코딩 → 정규화/표준화 → 모델 학습 → 평가 → 추가 모델 → 최종 선택 → 저장/새 예측 → Streamlit


## STEP 01. 실행 환경 확인


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

print("Python:", sys.executable)
print("Version:", sys.version.split()[0])
print("Working directory:", Path.cwd())
print("pandas / numpy / sklearn:", pd.__version__, np.__version__, sklearn.__version__)


## STEP 02. 데이터 로딩

`data/titanic/train.csv`가 없다면 저장소 루트에서 먼저 `python scripts/prepare_titanic_data.py`를 실행합니다.


In [ ]:
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

data_path = project_root / "data" / "titanic" / "train.csv"
if not data_path.is_file():
    raise FileNotFoundError(
        f"{data_path} 파일이 없습니다. 저장소 루트에서 "
        "python scripts/prepare_titanic_data.py 를 먼저 실행하세요."
    )

df = pd.read_csv(data_path)
display(df.head())
print("shape:", df.shape)


## STEP 03. 데이터 구조와 품질 확인


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_rate": (df.isna().mean() * 100).round(2),
    "nunique": df.nunique(dropna=False),
})
display(quality)
display(df.describe())

for col in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(f"\n[{col}]")
    display(df[col].value_counts(dropna=False).to_frame("count"))


## STEP 04. Target 정의

`Survived=1`을 Positive class로 두는 이진 분류 문제입니다.


In [ ]:
print(df["Survived"].value_counts(dropna=False).sort_index())
print({"task": "binary_classification", "target": "Survived", "positive_class": 1})


## STEP 05. 결측치 처리 원리 학습

여기서 만드는 `df_work`는 EDA용입니다. 모델링에서는 다시 원본 `df`에서 시작합니다.


In [ ]:
df_work = df.copy()

age_median_for_eda = df_work["Age"].median()
embarked_mode_for_eda = df_work["Embarked"].mode(dropna=True).iloc[0]

print("EDA Age median:", age_median_for_eda)
print("EDA Embarked mode:", embarked_mode_for_eda)

df_work["Age"] = df_work["Age"].fillna(age_median_for_eda)
df_work["Embarked"] = df_work["Embarked"].fillna(embarked_mode_for_eda)

display(df_work[["Age", "Embarked", "Cabin"]].isna().sum().to_frame("missing"))


## STEP 06. 컬럼 사용 정책 검토


In [ ]:
candidate_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
for col in candidate_cols:
    print(f"\n[{col}] dtype={df_work[col].dtype}, "
          f"nunique={df_work[col].nunique(dropna=False)}, "
          f"missing={df_work[col].isna().sum()}")
    print(df_work[col].dropna().astype(str).head(5).tolist())


## STEP 07. 범주형 인코딩 원리 학습

`df_encoded`는 인코딩 원리를 보기 위한 연습용이며 모델 입력으로 사용하지 않습니다.


In [ ]:
df_encoded = df_work.copy()
encoded_preview = pd.get_dummies(
    df_encoded[["Sex", "Embarked"]],
    columns=["Sex", "Embarked"],
    dtype=int,
)
display(encoded_preview.head())


## STEP 08. 시각화


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data=df_work, x="Survived")
plt.title("Survived distribution")
plt.show()

sns.barplot(data=df_work, x="Sex", y="Survived")
plt.title("Survival rate by Sex")
plt.show()

sns.barplot(data=df_work, x="Pclass", y="Survived")
plt.title("Survival rate by Pclass")
plt.show()

sns.boxplot(data=df_work, x="Survived", y="Fare")
plt.title("Fare by Survived")
plt.show()


## STEP 09. 기초 통계와 EDA


In [ ]:
sex_summary = (
    df_work.groupby("Sex", dropna=False)
    .agg(rows=("Survived", "size"), survival_rate=("Survived", "mean"))
)
pclass_summary = (
    df_work.groupby("Pclass", dropna=False)
    .agg(rows=("Survived", "size"), survival_rate=("Survived", "mean"))
)
sex_pclass_summary = (
    df_work.groupby(["Sex", "Pclass"], dropna=False)
    .agg(rows=("Survived", "size"), survival_rate=("Survived", "mean"))
)

display(sex_summary)
display(pclass_summary)
display(sex_pclass_summary)


### STEP 09-A. Fare 평균 차이 — Welch 독립표본 t-test

- H0: 생존자와 비생존자의 평균 Fare에는 차이가 없다.
- H1: 두 그룹의 평균 Fare에는 차이가 있다.
- `p-value < 0.05`이면 보통 H0를 기각합니다.


In [ ]:
from scipy.stats import ttest_ind

survived_fare = df_work.loc[df_work["Survived"] == 1, "Fare"].dropna()
not_survived_fare = df_work.loc[df_work["Survived"] == 0, "Fare"].dropna()

stat, p_value = ttest_ind(survived_fare, not_survived_fare, equal_var=False)

print("survived Fare mean:", round(survived_fare.mean(), 2), " n =", len(survived_fare))
print("not survived Fare mean:", round(not_survived_fare.mean(), 2), " n =", len(not_survived_fare))
print("t-statistic:", round(stat, 4))
print(f"p-value: {p_value:.3e}")


## STEP 10. Feature 설계

이번 기본 실행에서는 `FamilySize`, `IsAlone`을 직접 만들어 봅니다.


In [ ]:
feature_demo = df_work.copy()
feature_demo["FamilySize"] = feature_demo["SibSp"] + feature_demo["Parch"] + 1
feature_demo["IsAlone"] = (feature_demo["FamilySize"] == 1).astype(int)

display(feature_demo[["SibSp", "Parch", "FamilySize", "IsAlone", "Survived"]].head())


# Part 2. 모델링 — STEP 11–15

이제 `df_work`, `df_encoded`, `feature_demo`가 아니라 **원본 `df`에서 다시 시작**합니다.


## STEP 11. 학습/테스트 데이터 준비

Feature 생성부터 split까지 직접 실행합니다.


In [ ]:
model_source = df.copy()
model_source["FamilySize"] = model_source["SibSp"] + model_source["Parch"] + 1
model_source["IsAlone"] = (model_source["FamilySize"] == 1).astype(int)

raw_input_columns = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
feature_columns = raw_input_columns + ["FamilySize", "IsAlone"]
numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked", "IsAlone"]

X = model_source[feature_columns].copy()
y = model_source["Survived"].astype(int).copy()

print("X:", X.shape)
print("y:", y.shape)
display(X.head())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("train:", X_train.shape, y_train.shape)
print("test :", X_test.shape, y_test.shape)

print("전체 비율")
display(y.value_counts(normalize=True).sort_index().rename("ratio").to_frame())
print("학습 비율")
display(y_train.value_counts(normalize=True).sort_index().rename("train_ratio").to_frame())
print("테스트 비율")
display(y_test.value_counts(normalize=True).sort_index().rename("test_ratio").to_frame())


## STEP 12. 전처리를 하나씩 직접 실행하기

이번 STEP에서는 한꺼번에 묶지 않습니다.

1. 숫자형 결측치 처리
2. 범주형 결측치 처리
3. One-Hot Encoding
4. 정규화와 표준화 차이 확인
5. StandardScaler로 표준화
6. 숫자형과 범주형 데이터 결합
7. Logistic Regression 학습


### STEP 12-1. 숫자형 결측치 처리

중앙값은 **Train에서만 계산**하고 Test에는 같은 값을 적용합니다.


In [ ]:
from sklearn.impute import SimpleImputer

numeric_imputer = SimpleImputer(strategy="median")

X_train_num_imputed = pd.DataFrame(
    numeric_imputer.fit_transform(X_train[numeric_features]),
    columns=numeric_features,
    index=X_train.index,
)
X_test_num_imputed = pd.DataFrame(
    numeric_imputer.transform(X_test[numeric_features]),
    columns=numeric_features,
    index=X_test.index,
)

print("Train numeric missing:", X_train_num_imputed.isna().sum().sum())
print("Test numeric missing :", X_test_num_imputed.isna().sum().sum())
display(X_train_num_imputed.head())


### STEP 12-2. 범주형 결측치 처리

최빈값도 **Train에서만 학습**합니다.


In [ ]:
categorical_imputer = SimpleImputer(strategy="most_frequent")

X_train_cat_imputed = pd.DataFrame(
    categorical_imputer.fit_transform(X_train[categorical_features]),
    columns=categorical_features,
    index=X_train.index,
)
X_test_cat_imputed = pd.DataFrame(
    categorical_imputer.transform(X_test[categorical_features]),
    columns=categorical_features,
    index=X_test.index,
)

print("Train categorical missing:", X_train_cat_imputed.isna().sum().sum())
print("Test categorical missing :", X_test_cat_imputed.isna().sum().sum())
display(X_train_cat_imputed.head())


### STEP 12-3. One-Hot Encoding

범주 목록은 Train에서 학습하고 Test에는 같은 열 구조를 적용합니다.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat_encoded_array = encoder.fit_transform(X_train_cat_imputed)
X_test_cat_encoded_array = encoder.transform(X_test_cat_imputed)

encoded_feature_names = encoder.get_feature_names_out(categorical_features)

X_train_cat_encoded = pd.DataFrame(
    X_train_cat_encoded_array,
    columns=encoded_feature_names,
    index=X_train.index,
)
X_test_cat_encoded = pd.DataFrame(
    X_test_cat_encoded_array,
    columns=encoded_feature_names,
    index=X_test.index,
)

print("encoded columns:", list(encoded_feature_names))
display(X_train_cat_encoded.head())


### STEP 12-4. 정규화와 표준화 구분

- **정규화(Normalization)**: 값을 일정 범위로 맞춤. 대표적으로 `MinMaxScaler`를 사용하면 보통 0~1 범위가 됩니다.
- **표준화(Standardization)**: 평균 0, 표준편차 1을 기준으로 값을 바꿉니다. 대표적으로 `StandardScaler`를 사용합니다.

이번 Logistic Regression 실습에서는 **StandardScaler를 이용한 표준화**를 실제 모델 입력에 사용합니다. 먼저 Min-Max 정규화가 어떻게 보이는지만 간단히 확인합니다.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_demo = MinMaxScaler()
minmax_preview = pd.DataFrame(
    minmax_demo.fit_transform(X_train_num_imputed[["Age", "Fare"]]),
    columns=["Age_minmax", "Fare_minmax"],
    index=X_train.index,
)

display(minmax_preview.head())
print("이 결과는 개념 확인용이며 실제 모델 입력에는 사용하지 않습니다.")


### STEP 12-5. StandardScaler로 표준화

`fit_transform()`은 Train에, `transform()`은 Test에 사용합니다.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_num_imputed),
    columns=numeric_features,
    index=X_train.index,
)
X_test_num_scaled = pd.DataFrame(
    scaler.transform(X_test_num_imputed),
    columns=numeric_features,
    index=X_test.index,
)

display(X_train_num_scaled.head())
display(X_train_num_scaled.describe().loc[["mean", "std"]])


### STEP 12-6. 숫자형과 범주형 데이터 결합


In [ ]:
X_train_ready = pd.concat(
    [X_train_num_scaled, X_train_cat_encoded], axis=1
)
X_test_ready = pd.concat(
    [X_test_num_scaled, X_test_cat_encoded], axis=1
)

print("X_train_ready:", X_train_ready.shape)
print("X_test_ready :", X_test_ready.shape)
print("missing train:", X_train_ready.isna().sum().sum())
print("missing test :", X_test_ready.isna().sum().sum())
display(X_train_ready.head())


### STEP 12-7. Logistic Regression Baseline 학습


In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_ready, y_train)

print("model:", baseline_model)
print("classes:", baseline_model.classes_)


## STEP 13. 성능 평가와 Confusion Matrix


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

baseline_pred = baseline_model.predict(X_test_ready)

print("accuracy :", round(accuracy_score(y_test, baseline_pred), 4))
print("precision:", round(precision_score(y_test, baseline_pred), 4))
print("recall   :", round(recall_score(y_test, baseline_pred), 4))
print("f1       :", round(f1_score(y_test, baseline_pred), 4))
print()
print(classification_report(y_test, baseline_pred, digits=4))

cm = confusion_matrix(y_test, baseline_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:\n", cm)
print({"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)})


Confusion Matrix는 단순히 몇 개를 맞혔는지만 보는 것이 아니라 **어떤 종류의 실수(FP/FN)를 했는지** 보는 표입니다. 암 진단에서는 FN이 특히 중요할 수 있고, 스팸 분류에서는 정상 메일을 스팸으로 보내는 FP가 중요할 수 있습니다.


## STEP 14. 추가 모델 — Random Forest

Random Forest는 표준화가 필수는 아니지만, 이번에는 모델 비교를 단순하게 하기 위해 같은 준비 데이터를 사용합니다.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

additional_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
additional_model.fit(X_train_ready, y_train)
additional_pred = additional_model.predict(X_test_ready)

print("Random Forest accuracy:", round(accuracy_score(y_test, additional_pred), 4))
print("Random Forest f1      :", round(f1_score(y_test, additional_pred), 4))


## STEP 15. 모델 비교와 최종 모델 선택

이번 기본 실습에서는 같은 Test set의 결과를 비교해 최종 모델을 선택합니다. 더 엄밀한 실무에서는 별도 Validation set이나 Cross Validation을 추가할 수 있습니다.


In [ ]:
model_compare = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y_test, baseline_pred),
        "f1": f1_score(y_test, baseline_pred),
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(y_test, additional_pred),
        "f1": f1_score(y_test, additional_pred),
    },
])

display(model_compare)

FINAL_MODEL_CHOICE = "baseline"

if FINAL_MODEL_CHOICE == "baseline":
    final_model = baseline_model
    final_model_name = "Logistic Regression"
elif FINAL_MODEL_CHOICE == "random_forest":
    final_model = additional_model
    final_model_name = "Random Forest"
else:
    raise ValueError("FINAL_MODEL_CHOICE는 baseline 또는 random_forest여야 합니다.")

print("final model:", final_model_name)


# Part 3. 저장·새 입력 예측·서비스


## STEP 16. 전처리 객체와 모델 저장

한 개의 Pipeline을 저장하지 않고, 지금까지 직접 학습한 객체들을 각각 묶어 저장합니다.


In [ ]:
import json
import joblib

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

bundle_path = models_dir / "titanic_model_bundle.joblib"
contract_path = models_dir / "titanic_model_contract.json"

model_bundle = {
    "numeric_imputer": numeric_imputer,
    "categorical_imputer": categorical_imputer,
    "encoder": encoder,
    "scaler": scaler,
    "model": final_model,
}
joblib.dump(model_bundle, bundle_path)

model_contract = {
    "task": "binary_classification",
    "target": "Survived",
    "positive_class": 1,
    "raw_input_columns": raw_input_columns,
    "model_feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "prepared_feature_columns": X_train_ready.columns.tolist(),
    "derived_features": ["FamilySize", "IsAlone"],
    "scaling": "StandardScaler",
    "final_estimator": type(final_model).__name__,
}
contract_path.write_text(
    json.dumps(model_contract, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("saved bundle  :", bundle_path)
print("saved contract:", contract_path)


### STEP 16-A. 새로운 승객을 같은 순서로 직접 변환하고 예측


In [ ]:
loaded_bundle = joblib.load(bundle_path)

new_passenger = pd.DataFrame([{
    "Pclass": 3,
    "Sex": "male",
    "Age": 30.0,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 10.0,
    "Embarked": "S",
}])

new_passenger["FamilySize"] = new_passenger["SibSp"] + new_passenger["Parch"] + 1
new_passenger["IsAlone"] = (new_passenger["FamilySize"] == 1).astype(int)

new_num_imputed = loaded_bundle["numeric_imputer"].transform(new_passenger[numeric_features])
new_num_scaled = loaded_bundle["scaler"].transform(new_num_imputed)

new_cat_imputed = loaded_bundle["categorical_imputer"].transform(new_passenger[categorical_features])
new_cat_encoded = loaded_bundle["encoder"].transform(new_cat_imputed)

new_ready = pd.DataFrame(
    np.hstack([new_num_scaled, new_cat_encoded]),
    columns=model_contract["prepared_feature_columns"],
)

display(new_ready)

loaded_model = loaded_bundle["model"]
new_prediction = int(loaded_model.predict(new_ready)[0])
positive_index = list(loaded_model.classes_).index(1)
positive_probability = float(loaded_model.predict_proba(new_ready)[0][positive_index])

print("prediction:", new_prediction)
print("survival probability:", round(positive_probability, 4))


## STEP 17. Streamlit 서비스

Notebook에서 직접 확인한 순서를 앱에서도 그대로 사용합니다.

`원본 입력 → FamilySize/IsAlone → 결측치 처리 → One-Hot Encoding → StandardScaler → 모델 예측`

실행:

```powershell
streamlit run src/titanic_app/app.py
```


# 최종 정리

- STEP 07 인코딩은 원리 학습용입니다.
- STEP 11에서 원본 `df`로 다시 시작하고 먼저 Train/Test를 나눕니다.
- STEP 12에서 결측치 처리, One-Hot Encoding, 정규화/표준화를 각각 직접 실행합니다.
- `MinMaxScaler`는 정규화 개념 확인용이고 실제 기본 모델에는 `StandardScaler` 표준화를 사용합니다.
- Train에는 `fit_transform()`, Test에는 `transform()`을 사용해 데이터 누수를 막습니다.
- STEP 16에서는 전처리 객체와 모델을 따로 저장하고 새 승객에게 같은 순서로 적용합니다.
